# Reset the Workshop

**Reset** · under 1 min · Run any time

Clear Lab 2, clear Lab 1, or tear the whole thing down — run only the section you need.

**Run only the section you need. Do not use Run All** - section 3 drops the entire workshop, and on a Run All it would fire straight after the two partial resets.

The sections are ordered least to most destructive. Every statement uses `IF EXISTS`, so each one is safe to run twice and safe against a half-built environment.

**Before you start**, set the notebook's role to `SYSADMIN` and the warehouse to `GSI_TAX_WH`. If the warehouse has already been dropped, pick any other warehouse.

> Architecture, the business process behind it and how the notebooks fit together: **`gsi_tax_workshop_index.html`**. You do not need it open to work through this notebook - everything you need to run and interpret each step is here.

---

> ⚠️ **Synthetic data**
>
> All data in this workshop is synthetically generated for training purposes. Entity names, ABNs, suppliers and transactions are fictitious. Tax treatments shown are simplified for illustration and are not tax advice.

---

## Step 1 of 4 — Reset Lab 2 only, keep the ledger

**Lab 2 only** → Lab 1 only → Everything → Verify

*~20 sec* · [architecture & background](gsi_tax_workshop_index.html#step-1)

**Rebuilding 2.4 million ledger lines to re-test one AI function is a waste of your morning.** Lab 2 builds eight views, a semantic view, an AI Dynamic Table and an agent, and every one of them sits on top of data that has not changed. This clears Lab 2 and leaves the ledger alone, so you can re-run Lab 2 in about two minutes instead of five.

**Drop the two schemas rather than name the objects.** Lab 2's output lives entirely in `AI_INSIGHTS` and `AGENTS`, so dropping and recreating both is shorter than listing eight views and safer than trying to keep the list current as the lab changes. The agent is dropped explicitly first, because an agent created through Snowsight is not always removed with its schema.

In [ ]:
USE WAREHOUSE GSI_TAX_WH;
USE DATABASE GSI_TAX_WORKSHOP;

-- The agent first. It is not always removed with the schema it lives in.
DROP AGENT IF EXISTS AGENTS.GST_COMPLIANCE_AGENT;

-- Everything Lab 2 creates lives in these two schemas: the four AI views, the
-- AI Dynamic Table, the wide intelligence view, the supplier-risk view and the
-- semantic view. Dropping and recreating both is cleaner than naming them.
DROP SCHEMA IF EXISTS AI_INSIGHTS;
DROP SCHEMA IF EXISTS AGENTS;
CREATE SCHEMA AI_INSIGHTS;
CREATE SCHEMA AGENTS;

-- Confirm Lab 1 is untouched and Lab 2 is gone.
SELECT 'Lab 1 ledger'          AS LAYER,
       TO_VARCHAR(COUNT(*), '9,999,999') AS ROWS_PRESENT,
       'keep'                  AS EXPECTED
FROM CURATED.GL_HARMONISED
UNION ALL
SELECT 'Lab 2 objects',
       (SELECT COUNT(*)::VARCHAR FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA IN ('AI_INSIGHTS','AGENTS')),
       '0';

**The ledger still reads 2,400,000 and Lab 2 is empty.** Open `gsi_tax_lab2_cortex_ai.ipynb` and run it from the top. Nothing in setup or Lab 1 needs re-running.

---

## Step 2 of 4 — Reset Lab 1, keep the generated data

Lab 2 only → **Lab 1 only** → Everything → Verify

*~20 sec* · [architecture & background](gsi_tax_workshop_index.html#step-2)

**This clears everything Lab 1 built but keeps the 2.4 million rows setup generated.** The conformed ledger, the determination and reconciliation views, the exception register, the audit snapshots and the share all go. `RAW_DATA` and the two tax rule tables stay, so you can re-run Lab 1 without regenerating data.

**Two repairs matter more than the drops.** Lab 1 Step 5 posts a backdated adjustment with a fixed document number, and Step 4 raises the reverse-charge rate to 15% before putting it back. Neither is idempotent: run Lab 1 twice without clearing them and you stack a second adjustment onto the same entity-period, or leave the rate at 15% if a run stopped midway. Both are undone below, which is what makes a re-run behave like a first run.

In [ ]:
USE WAREHOUSE GSI_TAX_WH;
USE DATABASE GSI_TAX_WORKSHOP;

-- The share pins the database it grants on, so it always goes first.
DROP SHARE IF EXISTS GST_AUDIT_SNAPSHOT_SHARE;

-- The scenario clone from the optional modelling step.
DROP DATABASE IF EXISTS GSI_TAX_SCENARIO;

-- Lab 1's views. The two rule tables are NOT dropped: setup owns those, and the
-- whole point of this section is to avoid regenerating anything setup made.
DROP VIEW IF EXISTS TAX.V_GST_EXCEPTIONS;
DROP VIEW IF EXISTS TAX.V_GST_VS_BAS;
DROP VIEW IF EXISTS TAX.V_GST_DETERMINED;
DROP TABLE IF EXISTS TAX.STG_RULE_UPDATE;

-- The conformed ledger, the audit snapshots and the Streamlit app. Dropping the
-- schemas also clears the date-stamped snapshot clones without naming them.
DROP SCHEMA IF EXISTS CURATED;
DROP SCHEMA IF EXISTS AUDIT;
DROP SCHEMA IF EXISTS APPS;
CREATE SCHEMA CURATED;
CREATE SCHEMA AUDIT;
CREATE SCHEMA APPS;

-- Repair 1: remove the backdated adjustment so a re-run posts exactly one.
DELETE FROM RAW_DATA.ERP_SAP_GL WHERE BELNR = 'SAP9900001';

-- Repair 2: put the reverse-charge rate back to the Australian 10% in case a
-- run stopped between the two UPDATE statements in Step 4.
UPDATE TAX.TAX_RULE_RATE
   SET RATE_PCT = 10.00
 WHERE JURISDICTION = 'AU'
   AND TAX_TREATMENT = 'REVERSE_CHARGE'
   AND RATE_PCT <> 10.00;

-- Confirm setup survived and Lab 1 is gone.
SELECT 'SAP ledger rows'    AS CHECK_NAME, TO_VARCHAR(COUNT(*), '9,999,999') AS VALUE, '1,200,000' AS EXPECTED
FROM RAW_DATA.ERP_SAP_GL
UNION ALL
SELECT 'determination rules', COUNT(*)::VARCHAR, '16' FROM TAX.TAX_RULE_DETERMINATION
UNION ALL
SELECT 'AU reverse-charge rate',
       (SELECT LISTAGG(DISTINCT RATE_PCT::VARCHAR, ',') FROM TAX.TAX_RULE_RATE
        WHERE JURISDICTION = 'AU' AND TAX_TREATMENT = 'REVERSE_CHARGE'), '10.00'
UNION ALL
SELECT 'Lab 1 objects left',
       (SELECT COUNT(*)::VARCHAR FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA IN ('CURATED','AUDIT','APPS')), '0';

**1,200,000 SAP rows and 16 rules still there, the rate back at 10.00, and Lab 1's own objects gone.** Open `gsi_tax_lab1_foundation.ipynb` and run it from the top. This also clears Lab 2, because Lab 2 reads the views you just dropped — run Lab 1 first, then Lab 2.

---

## Step 3 of 4 — Drop everything

Lab 2 only → Lab 1 only → **Everything** → Verify

*~20 sec* · [architecture & background](gsi_tax_workshop_index.html#step-3)

**The full teardown: both labs, the setup data, the warehouse, everything.** Use this when you have finished with the environment, or when you want the next run to prove that setup works from nothing. Four statements remove every object this workshop created.

> ⚠️ **This is irreversible beyond the Time Travel window**
>
> Dropping a database starts its Time Travel clock, and `UNDROP DATABASE` works only until the retention period expires. Past that point the 2.4 million ledger lines, the rule tables and every view over them are gone with no recovery path. If you only want to stop the compute cost, run `ALTER WAREHOUSE GSI_TAX_WH SUSPEND;` instead and leave the databases in place — a suspended warehouse consumes no credits, and under a gigabyte of storage costs cents a month.

In [ ]:
-- Order matters. A share still granting on the database blocks the drop with
-- "cannot be dropped. It is still shared by 1 shares", so the share goes first.
DROP SHARE IF EXISTS GST_AUDIT_SNAPSHOT_SHARE;

-- The scenario clone from Lab 1 Step 7, if it is still around.
DROP DATABASE IF EXISTS GSI_TAX_SCENARIO;

-- The workshop itself: all seven schemas, both Dynamic Tables, every view, the
-- rule tables, the semantic view, the agent and the Streamlit app objects.
DROP DATABASE IF EXISTS GSI_TAX_WORKSHOP;

-- The warehouse last, so the statements above had compute to run on.
DROP WAREHOUSE IF EXISTS GSI_TAX_WH;

**Nothing named `GSI_TAX` is left in the account.** The next section confirms it. To rebuild from scratch, run `gsi_tax_lab0_setup.ipynb`, then Lab 1, then Lab 2.

---

## Step 4 of 4 — Verify what is left

Lab 2 only → Lab 1 only → Everything → **Verify**

*~10 sec* · [architecture & background](gsi_tax_workshop_index.html#step-4)

**Confirm what is actually left.** Whichever section you ran, this reports the state of the account so you are not guessing. It changes nothing, so it is safe to run at any time.

**Plain `SHOW`, with no `SELECT` wrapped around it.** The tidier version of this section counted the rows with `RESULT_SCAN`, which reads better but needs a warehouse — and after a full teardown there is no warehouse left, so the check that confirms the teardown could not run. `SHOW` is a metadata operation and needs no compute, which is exactly what a post-teardown verification requires. Zero rows means gone.

In [ ]:
-- Databases. After a full teardown: no rows. After a Lab 1 or Lab 2 reset:
-- GSI_TAX_WORKSHOP is still listed, which is correct.
SHOW TERSE DATABASES LIKE 'GSI_TAX%';

In [ ]:
-- The warehouse. Only the full teardown removes this one.
SHOW TERSE WAREHOUSES LIKE 'GSI_TAX%';

In [ ]:
-- The audit share. Every section above drops this, so it should always be empty.
-- A share left behind is the one leftover that blocks the database being dropped
-- later, which is why all three sections remove it.
SHOW SHARES LIKE 'GST_AUDIT%';

**Three grids, and what they should show depends on which section you ran.** After a full teardown all three are empty. After a Lab 1 or Lab 2 reset the database and warehouse are still there and only the share is gone. To rebuild from nothing, run `gsi_tax_lab0_setup.ipynb`, then Lab 1, then Lab 2.